# O-RAN Version 4 — Dual-Uncertainty QoS Safety-Shield Research Prototype

This notebook uses the complete Version 4 source package committed in this repository. It bootstraps the package, runs software checks and executes a small end-to-end integration experiment. The small run is for reproducibility and software verification, not for scientific performance claims.

The model is a single-cell simulation prototype. It is not a live O-RAN deployment and does not provide a formal or simultaneous QoS safety guarantee.

In [ ]:
from pathlib import Path
import os, sys, tarfile

REPO_ROOT = Path.cwd().resolve()
ARCHIVE = REPO_ROOT / "source_packages" / "O-RAN-Version4-Dual-Uncertainty-QoS-Safe-DRL.tar.gz"
V4 = REPO_ROOT / "experiments" / "AI-Enabled-O-RAN-ML-DRL-QoS-Optimization-Version4"
if not (V4 / "run_all.py").exists():
    if not ARCHIVE.exists():
        raise FileNotFoundError("Open this notebook from the repository root so source_packages/ is available.")
    destination = REPO_ROOT / "experiments"
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE, "r:gz") as bundle:
        root = destination.resolve()
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe archive path: {member.name}")
        bundle.extractall(destination)
if not (V4 / "run_all.py").exists():
    raise FileNotFoundError(f"Version 4 source package not found at {V4}")
os.chdir(V4)
sys.path.insert(0, str(V4))
print(f"Using source package: {V4}")

## Install dependencies and run software checks

The following cell compiles the complete source tree and runs the six unit tests. A passing test suite checks implementation consistency; it does not validate research performance.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "compileall", "-q", "config.py", "src", "tests", "run_all.py"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## Run the complete pipeline — software integration configuration

This compact configuration trains the traffic model and separate MRC, ZF and MMSE PPO agents, performs channel calibration, and evaluates the methods. It is intended to verify that the workflow runs from beginning to end.

In [ ]:
smoke_command = [
    sys.executable, "run_all.py",
    "--seeds", "2", "--blocks", "3",
    "--traffic-epochs", "2",
    "--calibration-samples", "20",
    "--ppo-episodes", "2", "--ppo-steps", "3",
]
subprocess.run(smoke_command, check=True)
summary_file = Path("results/summary_ci95.csv")
if summary_file.exists():
    import pandas as pd
    display(pd.read_csv(summary_file))
else:
    print("Pipeline completed. Inspect the generated results/ and artifacts/ directories.")

## Scaling to research experiments

The package README documents a larger starting configuration: 10 independent evaluation seeds, 200 blocks per seed, 100 traffic-model epochs, 500 independent channel-calibration units, and 250 PPO episodes × 40 steps per receiver. This is a starting point, not a guarantee of sufficient statistical power. Use independent training, calibration, validation and test seeds; report intervals and method-specific QoS violation rates.

After training and calibration, a traffic/CSI stress evaluation can be run with:

```bash
python -m src.evaluate --seeds 10 --blocks 200 --csi-error-multiplier 1.5 --traffic-surge-prob 0.10
```

## Research limitations

- The demand MLP did not outperform the persistence baseline in the reported small integration run.
- The calibration design uses one independent channel realization as a calibration unit and a finite predeclared action family. It does not guarantee coverage for arbitrary SLSQP-projected actions, adaptive policies, distribution shifts or simultaneous all-user safety.
- SLSQP solves a non-convex projection problem locally and can fail. The equal-power fallback preserves the sum-power budget only; QoS is not guaranteed.
- The traffic, channel and power models are simulated. There is no live A1/E2 interface, deployed xApp/rApp or measured RAN energy consumption.

See `README.md`, `docs/V4_RESEARCH_NOVELTY.md` and `docs/V4_VALIDATION_REPORT.md` for assumptions, candidate research questions and validation gaps.